In [69]:
# Set seed for reproducibility
SEED = 16

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
# from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import TensorDataset, DataLoader
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna


# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
PyTorch version: 2.6.0+cu124
Device: cuda


In [15]:
# Uninstall any existing conflicting versions
!pip uninstall tensorboard -y
!pip uninstall tensorflow -y
!pip uninstall tensorflow-estimator -y

Found existing installation: tensorboard 2.20.0
Uninstalling tensorboard-2.20.0:
  Successfully uninstalled tensorboard-2.20.0
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0


In [17]:
# Reinstall a recent, known good version of tensorboard and the necessary TensorFlow packages
!pip install tensorboard tensorflow

  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorflow-2.20.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
Using cached tensorboard-2.20.0-py3-none-any.whl (5.5 MB)
Using cached tensorflow-2.20.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (620.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
tf-keras 2.18.0 requires tensorflow<2.19,>=2.18, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.20.0 which is incompatible.
tensorflow-text 2.18.1 requires tensorflow<2.19,>=2.18.0, but you have tensorflow 2.20.0 which is incompatible.


In [70]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/the-pirate-pain-dataset/sample_submission.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_test.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_train_labels.csv
/kaggle/input/the-pirate-pain-dataset/pirate_pain_train.csv


## ⏳ **Data Loading**

## 🔎 **Exploration and Data Analysis**

In [71]:
# Load the dataset from a CSV file
df_train = pd.read_csv("/kaggle/input/the-pirate-pain-dataset/pirate_pain_train.csv")
df_public_test = pd.read_csv("/kaggle/input/the-pirate-pain-dataset/pirate_pain_test.csv")
df_labels = pd.read_csv("/kaggle/input/the-pirate-pain-dataset/pirate_pain_train_labels.csv")

# Print the shape of the DataFrame
# print(f"DataFrame sh/kaggle/input/the-pirate-pain-dataset/sample_submission.csv

## 🔄 **Data Preprocessing**

In [72]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_labels['label_encoded'] = label_encoder.fit_transform(df_labels['label'])
# This gives: ['high_pain', 'low_pain', 'no_pain']
label_map = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2}
df_labels['label_encoded'] = df_labels['label'].map(label_map)

In [73]:
from sklearn.preprocessing import MinMaxScaler
# Convert static cols
def preProcess(df):
    # Changed from .min(axis=1) to .median(axis=1)
    df["pain_survey"] = np.floor(df[["pain_survey_1", "pain_survey_2", "pain_survey_3", "pain_survey_4"]].median(axis=1)).astype(int) 
    df.drop(columns=['pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4'], inplace=True)
    
    df['merged_n_features'] = np.where(
        (df['n_legs'] == 'two') & (df['n_hands'] == 'two') & (df['n_eyes'] == 'two'),
        0, 
        1  
    )
    
    # Calculate the count of rows where n_legs, n_hands, or n_eyes are not 'two'
    explicit_different_count = df[
        (df['n_legs'] != 'two') | (df['n_hands'] != 'two') | (df['n_eyes'] != 'two')
    ].shape[0]
    
    df.drop(columns=['n_legs', 'n_hands', 'n_eyes', "joint_30"], inplace=True)
    
    columns_to_scale = [col for col in df.columns if col not in ['time', 'sample_index']]

    scaler = MinMaxScaler()
    df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

preProcess(df_train)
preProcess(df_public_test)

df_train.head()

,sample_index,time,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,...,joint_22,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,pain_survey,merged_n_features
0,0,0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,...,1.374706e-06,0.000015,3.162813e-04,0.000004,0.014214,0.011376,0.018978,0.020291,0.5,0.0
1,0,1,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654605,0.760554,...,4.026521e-07,0.000022,9.828599e-07,0.000000,0.010748,0.000000,0.009473,0.010006,1.0,0.0
2,0,2,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,...,1.440847e-08,0.000005,6.626013e-05,0.000003,0.013097,0.006830,0.017065,0.016856,1.0,0.0
3,0,3,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,...,3.065580e-07,0.000007,1.199337e-06,0.000000,0.009505,0.006274,0.020264,0.017981,1.0,0.0
4,0,4,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,...,1.723863e-08,0.000006,1.307199e-06,0.000007,0.004216,0.002132,0.023389,0.018477,1.0,0.0


In [77]:
from sklearn.preprocessing import MinMaxScaler
# Scaling
scale_cols = [f'joint_{i:02d}' for i in range(30)] + ['pain_survey', 'merged_n_features']

scaler = MinMaxScaler()

# FIT the scaler ONLY on the training data
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])

# TRANSFORM the test data with the same scaler
df_public_test[scale_cols] = scaler.transform(df_public_test[scale_cols])

In [78]:
from sklearn.model_selection import train_test_split
# STRATIFICATION
user_labels = df_labels.copy()

train_users, val_users = train_test_split(
    user_labels['sample_index'],
    test_size=0.2,  # 20% validation
    stratify=user_labels['label'], # This is the key for imbalance
    random_state=SEED
)

df_train_full = df_train.copy()

# Filter the full dataframe to get rows for train/val users
df_train_fold = df_train_full[df_train_full['sample_index'].isin(train_users)]
df_val_fold   = df_train_full[df_train_full['sample_index'].isin(val_users)]

# Merge labels back in
df_train_fold = df_train_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')
df_val_fold   = df_val_fold.merge(user_labels[['sample_index', 'label', 'label_encoded']], on='sample_index', how='left')

# Check class distribution
print("--- Training Set Class Distribution ---")
print(df_train_fold['label'].value_counts(normalize=True))
print("\n--- Validation Set Class Distribution ---")
print(df_val_fold['label'].value_counts(normalize=True))

df_train_fold = df_train_fold.drop(columns=['label'])
df_val_fold = df_val_fold.drop(columns=['label'])
print(df_train_fold.head())

# X_train_final = df_train_fold.drop(columns=['label', 'label_encoded'])
# y_train_final = df_train_fold[['sample_index', 'label_encoded']]

# X_val_final = df_val_fold.drop(columns=['label', 'label_encoded'])
# y_val_final = df_val_fold[['sample_index', 'label_encoded']]

--- Training Set Class Distribution ---
label
no_pain      0.772727
low_pain     0.142045
high_pain    0.085227
Name: proportion, dtype: float64

--- Validation Set Class Distribution ---
label
no_pain      0.774436
low_pain     0.142857
high_pain    0.082707
Name: proportion, dtype: float64
   sample_index  time  joint_00  joint_01  joint_02  joint_03  joint_04  \
0             0     0  0.777507  0.738252  0.779512  0.804419  0.714916   
1             0     1  0.806256  0.765147  0.761153  0.838021  0.735684   
2             0     2  0.767592  0.721439  0.772834  0.777832  0.724497   
3             0     3  0.666220  0.810416  0.763971  0.785928  0.679928   
4             0     4  0.774297  0.773366  0.772162  0.767017  0.747710   

   joint_05  joint_06  joint_07  ...  joint_23      joint_24  joint_25  \
0  0.736643  0.639301  0.733981  ...  0.000015  3.162813e-04  0.000004   
1  0.729533  0.654605  0.760554  ...  0.000022  9.828599e-07  0.000000   
2  0.734962  0.692340  0.787647  .

In [79]:
# Convert all feature columns (X) to float32
# This includes the scaled dynamic cols and the static/numeric pirate features
df_train_fold = df_train_fold.astype('float32')
df_val_fold   = df_val_fold.astype('float32')

# Convert the target label column (y) to int64
# PyTorch's CrossEntropyLoss expects class labels as LongTensors (int64)
df_train_fold['label_encoded']  = df_train_fold['label_encoded'].astype('int64')
df_val_fold['label_encoded']   = df_val_fold['label_encoded'].astype('int64')


# --- Verify the changes ---
print("--- df_train_fold (Features) dtypes ---")
# .info() will show all columns are now float32
print(df_train_fold.info())

--- df_train_fold (Features) dtypes ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84480 entries, 0 to 84479
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sample_index       84480 non-null  float32
 1   time               84480 non-null  float32
 2   joint_00           84480 non-null  float32
 3   joint_01           84480 non-null  float32
 4   joint_02           84480 non-null  float32
 5   joint_03           84480 non-null  float32
 6   joint_04           84480 non-null  float32
 7   joint_05           84480 non-null  float32
 8   joint_06           84480 non-null  float32
 9   joint_07           84480 non-null  float32
 10  joint_08           84480 non-null  float32
 11  joint_09           84480 non-null  float32
 12  joint_10           84480 non-null  float32
 13  joint_11           84480 non-null  float32
 14  joint_12           84480 non-null  float32
 15  joint_13           84480 non-n

In [80]:
df_train_fold.head(-10)

,sample_index,time,joint_00,joint_01,joint_02,joint_03,joint_04,joint_05,joint_06,joint_07,...,joint_23,joint_24,joint_25,joint_26,joint_27,joint_28,joint_29,pain_survey,merged_n_features,label_encoded
0,0.0,0.0,0.777507,0.738252,0.779512,0.804419,0.714916,0.736643,0.639301,0.733981,...,1.458410e-05,3.162813e-04,0.000004,0.014214,0.011376,0.018978,0.020291,0.5,0.0,0
1,0.0,1.0,0.806256,0.765147,0.761153,0.838021,0.735684,0.729533,0.654604,0.760554,...,2.195013e-05,9.828599e-07,0.000000,0.010748,0.000000,0.009473,0.010006,1.0,0.0,0
2,0.0,2.0,0.767592,0.721439,0.772834,0.777832,0.724497,0.734962,0.692340,0.787647,...,5.272918e-06,6.626013e-05,0.000003,0.013097,0.006830,0.017065,0.016856,1.0,0.0,0
3,0.0,3.0,0.666220,0.810416,0.763971,0.785928,0.679928,0.722504,0.589127,0.793524,...,6.737126e-06,1.199337e-06,0.000000,0.009505,0.006274,0.020264,0.017981,1.0,0.0,0
4,0.0,4.0,0.774297,0.773366,0.772162,0.767017,0.747710,0.743156,0.677764,0.725437,...,5.661887e-06,1.307199e-06,0.000007,0.004216,0.002132,0.023389,0.018477,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84465,660.0,145.0,0.747113,0.633353,0.556208,0.526835,0.058241,0.129760,0.648898,0.600525,...,3.068689e-06,6.187494e-06,0.000002,0.057496,0.008263,0.121502,0.054748,1.0,0.0,0
84466,660.0,146.0,0.766650,0.730096,0.596908,0.583812,0.098994,0.119217,0.706742,0.624951,...,1.851797e-06,6.117617e-06,0.000001,0.042580,0.019101,0.195717,0.102979,1.0,0.0,0
84467,660.0,147.0,0.746536,0.704494,0.650648,0.585991,0.064944,0.182338,0.678559,0.596394,...,5.101990e-06,6.046634e-06,0.000001,0.036764,0.012704,0.125387,0.075700,1.0,0.0,0
84468,660.0,148.0,0.719192,0.599540,0.572594,0.605968,0.097262,0.170702,0.676817,0.604818,...,1.269752e-07,5.974558e-06,0.000008,0.015018,0.008760,0.137120,0.082437,1.0,0.0,0


## Prepare data for training

In [81]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [82]:
feature_cols = [col for col in df_train.columns if col not in ["sample_index", "time"]]

In [83]:
WINDOW = 40
STRIDE = 10

BATCH_SIZE = 64
num_features = len(feature_cols)

In [84]:
import numpy as np

def build_sequences(df, feature_cols, id_col='sample_index', label_col='label_encoded', window=200, stride=200):
    """
    Builds sequences from a time-series dataframe.
    
    Args:
        df (pd.DataFrame): The input DataFrame (e.g., df_train_fold) containing
                           features, IDs, and labels.
        feature_cols (list): A list of column names to be used as features.
        id_col (str): The name of the column for unique sample IDs.
        label_col (str): The name of the column for the labels.
        window (int): The size of each sequence (window).
        stride (int): The step size between sequences.
    """
    # Sanity check
    # assert window % stride == 0
    
    num_features = len(feature_cols)
    dataset = []
    labels = []

    # Iterate over unique sample IDs
    for sample_id in df[id_col].unique():
        
        # Get the dataframe for the current sample
        temp_df = df[df[id_col] == sample_id]

        # Extract feature data for the current ID
        temp_features = temp_df[feature_cols].values

        # Retrieve the single label for the current ID
        # (Assumes all rows for one ID have the same label)
        label = temp_df[label_col].values[0]

        # Calculate padding length to ensure full windows
        # This logic correctly handles cases where length is already a multiple
        padding_len = (window - len(temp_features) % window) % window
        
        if padding_len > 0:
            # Create zero padding and concatenate with the data
            padding = np.zeros((padding_len, num_features), dtype='float32')
            temp_features = np.concatenate((temp_features, padding))

        # Build feature windows and associate them with labels
        idx = 0
        while idx + window <= len(temp_features):
            dataset.append(temp_features[idx:idx + window])
            labels.append(label)
            idx += stride

    # Convert lists to numpy arrays for further processing
    dataset = np.array(dataset)
    labels = np.array(labels)

    return dataset, labels

In [85]:
# Generate sequences and labels for the training set
X_train, y_train = build_sequences(
    df_train_fold, 
    feature_cols=feature_cols, 
    id_col='sample_index',
    label_col='label_encoded',
    window=WINDOW, 
    stride=STRIDE
)

# Generate sequences and labels for the validation set
X_val, y_val = build_sequences(
    df_val_fold, 
    feature_cols=feature_cols,
    id_col='sample_index',
    label_col='label_encoded',
    window=WINDOW, 
    stride=STRIDE
)

# Print the shapes of the generated datasets and their labels
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((6864, 40, 32), (6864,), (1729, 40, 32), (1729,))

In [86]:
# Define the input shape based on the training data
input_shape = X_train.shape[1:]

num_classes = 3

In [87]:
# Convert numpy arrays to PyTorch datasets (pairs features with labels)
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds   = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

In [88]:
def make_loader(ds, batch_size, shuffle, drop_last):
    # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    # Create DataLoader with performance optimizations
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,  # Faster GPU transfer
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
        prefetch_factor=4,  # Load 4 batches ahead
    )

In [89]:
# Create data loaders with different settings for each phase
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [91]:
import math
import torch
from torch.optim import Optimizer

class Ranger(Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.5, k=6, betas=(0.95, 0.999), eps=1e-5, weight_decay=0):
        """
        Ranger = RAdam + Lookahead
        Args:
            params: model parameters
            lr: learning rate
            alpha: lookahead step size (0.5 is default)
            k: lookahead steps before sync (6 is default)
            betas: RAdam betas
            eps: numerical stability
            weight_decay: L2 regularization
        """
        defaults = dict(lr=lr, alpha=alpha, k=k, betas=betas, eps=eps, weight_decay=weight_decay)
        super(Ranger, self).__init__(params, defaults)
        self._step = 0

        for group in self.param_groups:
            group["slow_params"] = [p.clone().detach() for p in group["params"] if p.requires_grad]

    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            for p, sp in zip(group["params"], group["slow_params"]):
                if p.grad is None:
                    continue

                grad = p.grad.data
                if grad.is_sparse:
                    raise RuntimeError("Ranger does not support sparse gradients")

                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p.data)
                    state["exp_avg_sq"] = torch.zeros_like(p.data)

                exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                beta1, beta2 = group["betas"]

                state["step"] += 1
                self._step += 1

                # Apply weight decay
                if group["weight_decay"] != 0:
                    grad = grad.add(p.data, alpha=group["weight_decay"])

                # Update exponential moving averages
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                # Compute rectified term (RAdam)
                bias_correction1 = 1 - beta1 ** state["step"]
                bias_correction2 = 1 - beta2 ** state["step"]
                n_sma_max = 2 / (1 - beta2) - 1
                n_sma = n_sma_max - 2 * state["step"] * (beta2 ** state["step"]) / bias_correction2

                if n_sma >= 5:
                    step_size = group["lr"] * math.sqrt(
                        ((1 - beta2 ** state["step"]) * (n_sma - 4) / (n_sma_max - 4)) *
                        ((n_sma - 2) / n_sma) * (n_sma_max / (n_sma_max - 2))
                    ) / bias_correction1
                    denom = exp_avg_sq.sqrt().add_(group["eps"])
                    p.data.addcdiv_(exp_avg, denom, value=-step_size)
                else:
                    step_size = group["lr"] / bias_correction1
                    p.data.add_(exp_avg, alpha=-step_size)

                # Lookahead updates
                if self._step % group["k"] == 0:
                    sp.add_(p.data - sp, alpha=group["alpha"])
                    p.data.copy_(sp)

        return loss


## 🛠️ **Model Building**

In [92]:
def recurrent_summary(model, input_size):
    """
    Custom summary function that emulates torchinfo's output while correctly
    counting parameters for RNN/GRU/LSTM layers.

    This function is designed for models whose direct children are
    nn.Linear, nn.RNN, nn.GRU, or nn.LSTM layers.

    Args:
        model (nn.Module): The model to analyze.
        input_size (tuple): Shape of the input tensor (e.g., (seq_len, features)).
    """

    # Dictionary to store output shapes captured by forward hooks
    output_shapes = {}
    # List to track hook handles for later removal
    hooks = []

    def get_hook(name):
        """Factory function to create a forward hook for a specific module."""
        def hook(module, input, output):
            # Handle RNN layer outputs (returns a tuple)
            if isinstance(output, tuple):
                # output[0]: all hidden states with shape (batch, seq_len, hidden*directions)
                shape1 = list(output[0].shape)
                shape1[0] = -1  # Replace batch dimension with -1

                # output[1]: final hidden state h_n (or tuple (h_n, c_n) for LSTM)
                if isinstance(output[1], tuple):  # LSTM case: (h_n, c_n)
                    shape2 = list(output[1][0].shape)  # Extract h_n only
                else:  # RNN/GRU case: h_n only
                    shape2 = list(output[1].shape)

                # Replace batch dimension (middle position) with -1
                shape2[1] = -1

                output_shapes[name] = f"[{shape1}, {shape2}]"

            # Handle standard layer outputs (e.g., Linear)
            else:
                shape = list(output.shape)
                shape[0] = -1  # Replace batch dimension with -1
                output_shapes[name] = f"{shape}"
        return hook

    # 1. Determine the device where model parameters reside
    try:
        device = next(model.parameters()).device
    except StopIteration:
        device = torch.device("cpu")  # Fallback for models without parameters

    # 2. Create a dummy input tensor with batch_size=1
    dummy_input = torch.randn(1, *input_size).to(device)

    # 3. Register forward hooks on target layers
    # Iterate through direct children of the model (e.g., self.rnn, self.classifier)
    for name, module in model.named_children():
        if isinstance(module, (nn.Linear, nn.RNN, nn.GRU, nn.LSTM)):
            # Register the hook and store its handle for cleanup
            hook_handle = module.register_forward_hook(get_hook(name))
            hooks.append(hook_handle)

    # 4. Execute a dummy forward pass in evaluation mode
    model.eval()
    with torch.no_grad():
        try:
            model(dummy_input)
        except Exception as e:
            print(f"Error during dummy forward pass: {e}")
            # Clean up hooks even if an error occurs
            for h in hooks:
                h.remove()
            return

    # 5. Remove all registered hooks
    for h in hooks:
        h.remove()

    # --- 6. Print the summary table ---

    print("-" * 79)
    # Column headers
    print(f"{'Layer (type)':<25} {'Output Shape':<28} {'Param #':<18}")
    print("=" * 79)

    total_params = 0
    total_trainable_params = 0

    # Iterate through modules again to collect and display parameter information
    for name, module in model.named_children():
        if name in output_shapes:
            # Count total and trainable parameters for this module
            module_params = sum(p.numel() for p in module.parameters())
            trainable_params = sum(p.numel() for p in module.parameters() if p.requires_grad)

            total_params += module_params
            total_trainable_params += trainable_params

            # Format strings for display
            layer_name = f"{name} ({type(module).__name__})"
            output_shape_str = str(output_shapes[name])
            params_str = f"{trainable_params:,}"

            print(f"{layer_name:<25} {output_shape_str:<28} {params_str:<15}")

    print("=" * 79)
    print(f"Total params: {total_params:,}")
    print(f"Trainable params: {total_trainable_params:,}")
    print(f"Non-trainable params: {total_params - total_trainable_params:,}")
    print("-" * 79)

In [93]:
class RecurrentClassifier(nn.Module):
    """
    Generic RNN classifier (RNN, LSTM, GRU).
    Uses the last hidden state for classification.
    """
    def __init__(
            self,
            input_size,
            hidden_size,
            num_layers,
            num_classes,
            rnn_type='GRU',        # 'RNN', 'LSTM', or 'GRU'
            bidirectional=False,
            dropout_rate=0.2
            ):
        super().__init__()

        self.rnn_type = rnn_type
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.bidirectional = bidirectional

        # Map string name to PyTorch RNN class
        rnn_map = {
            'RNN': nn.RNN,
            'LSTM': nn.LSTM,
            'GRU': nn.GRU
        }

        if rnn_type not in rnn_map:
            raise ValueError("rnn_type must be 'RNN', 'LSTM', or 'GRU'")

        rnn_module = rnn_map[rnn_type]

        # Dropout is only applied between layers (if num_layers > 1)
        dropout_val = dropout_rate if num_layers > 1 else 0

        # Create the recurrent layer
        self.rnn = rnn_module(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # Input shape: (batch, seq_len, features)
            bidirectional=bidirectional,
            dropout=dropout_val
        )

        # Calculate input size for the final classifier
        if self.bidirectional:
            classifier_input_size = hidden_size * 2 # Concat fwd + bwd
        else:
            classifier_input_size = hidden_size

        # Final classification layer
        self.classifier = nn.Linear(classifier_input_size, num_classes)

    def forward(self, x):
        """
        x shape: (batch_size, seq_length, input_size)
        """

        # rnn_out shape: (batch_size, seq_len, hidden_size * num_directions)
        rnn_out, hidden = self.rnn(x)

        # LSTM returns (h_n, c_n), we only need h_n
        if self.rnn_type == 'LSTM':
            hidden = hidden[0]

        # hidden shape: (num_layers * num_directions, batch_size, hidden_size)

        if self.bidirectional:
            # Reshape to (num_layers, 2, batch_size, hidden_size)
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)

            # Concat last fwd (hidden[-1, 0, ...]) and bwd (hidden[-1, 1, ...])
            # Final shape: (batch_size, hidden_size * 2)
            hidden_to_classify = torch.cat([hidden[-1, 0, :, :], hidden[-1, 1, :, :]], dim=1)
        else:
            # Take the last layer's hidden state
            # Final shape: (batch_size, hidden_size)
            hidden_to_classify = hidden[-1]

        # Get logits
        logits = self.classifier(hidden_to_classify)
        return logits

In [94]:
def initialize_weights(model, init_scheme='xavier_uniform'):
    """Initializes weights using the specified scheme."""
    for name, param in model.named_parameters():
        if 'weight_ih' in name or 'weight_hh' in name:
            # Recurrent weights
            if init_scheme == 'xavier_uniform':
                torch.nn.init.xavier_uniform_(param.data)
            elif init_scheme == 'orthogonal':
                # Often preferred for RNNs to maintain gradient scale
                torch.nn.init.orthogonal_(param.data)
        elif 'weight' in name:
            # Linear or Convolutional weights (e.g., in the final layer)
            if init_scheme == 'xavier_uniform':
                torch.nn.init.xavier_uniform_(param.data)
            elif init_scheme == 'kaiming_normal':
                 # Good for ReLU/Leaky ReLU (common in surrounding layers)
                torch.nn.init.kaiming_normal_(param.data, mode='fan_in', nonlinearity='relu')
        elif 'bias' in name:
            # Set biases to zero, which is common
            torch.nn.init.constant_(param.data, 0)
            
            # Optional: initialize forget gate bias to a positive value (e.g., 1.0 or 2.0)
            if 'bias_hh' in name and 'LSTM' in name: 
                 n = param.shape[0]
                 start, end = n // 4, n // 2
                 param.data[start:end].fill_(1.0) # For LSTM forget gate bias

In [95]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for multi-class classification.
    Reference: https://arxiv.org/abs/1708.02002 (Lin et al. 2017)

    Args:
        alpha (float or list): Weighting factor for classes (balances class imbalance)
        gamma (float): Focusing parameter to reduce the loss for well-classified examples
        reduction (str): 'none' | 'mean' | 'sum'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        if isinstance(alpha, (float, int)):
            self.alpha = torch.tensor([alpha])
        else:
            self.alpha = torch.tensor(alpha) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        Args:
            inputs: Predictions (logits), shape [batch_size, num_classes]
            targets: Ground truth labels, shape [batch_size]
        """
        # Compute log-probabilities
        log_probs = F.log_softmax(inputs, dim=1)
        probs = torch.exp(log_probs)

        # Select log-probability of the correct class
        log_probs_true = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        probs_true = probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        # Compute focal weight
        focal_weight = (1 - probs_true) ** self.gamma

        # Apply alpha (class weight)
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_factor = self.alpha[targets]
            focal_weight = alpha_factor * focal_weight

        # Compute final loss
        loss = -focal_weight * log_probs_true

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss


## 🧮 **Network and Training Hyperparameters**

In [96]:
from sklearn.utils.class_weight import compute_class_weight

# --- Calculate Class Weights ---
labels = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=labels,
    y=y_train
)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"Original label counts: {np.bincount(y_train)}")
print(f"Calculated weights: {weights_tensor}")

Original label counts: [5304  975  585]
Calculated weights: tensor([0.4314, 2.3467, 3.9111], device='cuda:0')


In [56]:
# # Training configuration
# RNN_TYPE = 'GRU'
# BIDIRECTIONAL = True

# LEARNING_RATE = 1e-4
# EPOCHS = 100
# PATIENCE = 30

# # Architecture
# HIDDEN_LAYERS = 2        # Hidden layers
# HIDDEN_SIZE = 128        # Neurons per layer

# # Regularisation
# DROPOUT_RATE = 0.4         # Dropout probability
# L1_LAMBDA = 0            # L1 penalty
# L2_LAMBDA = 1e-4            # L2 penalty

# # Set up loss function and optimizer
# criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)

In [101]:
# Assuming necessary imports (torch, optuna, FocalLoss, RecurrentClassifier, initialize_weights, 
# build_sequences, make_loader, TensorDataset, etc.) and global variables (BATCH_SIZE, 
# df_train_fold, df_val_fold, feature_cols, weights_tensor, input_shape, num_classes, device)
# are defined elsewhere in your Kaggle notebook.

# --- 1. Define Global Training Constants ---
EPOCHS = 200
PATIENCE = 15 # Used as the maximum patience for Early Stopping, 
              # but now we will tune a separate patience for the LR scheduler.

# --- 2. Define the Optuna Objective Function ---
def objective(trial):
    
    # --- 1. Tune Window and Stride (Data Generation) ---
    # Expanded range for better exploration
    window_hp = trial.suggest_categorical("WINDOW", [20, 40])
    stride_hp = trial.suggest_categorical("STRIDE", [10, 15, 20, 30])

    # Prune if stride is larger than the window
    if stride_hp >= window_hp:
        raise optuna.exceptions.TrialPruned()

    # --- 2. Build Dynamic Datasets and Loaders ---
    # Generate sequences for this trial
    X_train, y_train = build_sequences(
        df_train_fold, 
        feature_cols=feature_cols, 
        id_col='sample_index',
        label_col='label_encoded',
        window=window_hp, 
        stride=stride_hp
    )
    X_val, y_val = build_sequences(
        df_val_fold, 
        feature_cols=feature_cols,
        id_col='sample_index',
        label_col='label_encoded',
        window=window_hp, 
        stride=stride_hp
    )

    # Create TensorDatasets for this trial
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds   = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

    # Create DataLoaders for this trial
    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = make_loader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    
    # --- 3. Suggest Other Hyperparameters ---
    
    hidden_size_hp = trial.suggest_categorical("hidden_size", [32, 64, 128]) # Reduced range
    num_layers_hp = trial.suggest_int("num_layers", 1, 3)
    dropout_hp = trial.suggest_float("dropout_rate", 0.1, 0.7) # Increased max
    rnn_type_hp = trial.suggest_categorical("rnn_type", ["GRU"])
    bidirectional_hp = trial.suggest_categorical("bidirectional", [True, False])
    
    # Initialization (SYNTAX FIX: Removed the extra closing parenthesis)
    init_scheme_hp = trial.suggest_categorical(
        "init_scheme", 
        ["xavier_uniform", "orthogonal", "kaiming_normal"]
    )
    
    # Optimization Parameters (L2)
    lr_hp = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay_hp = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    
    # NEW L1 Regularization (For sparsity/simplification)
    l1_lambda_hp = trial.suggest_float("l1_lambda", 1e-7, 1e-4, log=True)
    
    # Focal Loss Parameters (gamma=0.0 now includes CrossEntropy)
    focal_gamma_hp = trial.suggest_float("focal_gamma", 0.0, 5.0, step=0.5)
    
    # Scheduler Parameters
    scheduler_patience_hp = trial.suggest_int("scheduler_patience", 3, 10, step=1)
    scheduler_factor_hp = trial.suggest_categorical("scheduler_factor", [0.1, 0.2, 0.5])
    
    # --- 4. Create Model, Criterion, Optimizer, and Scheduler ---
    
    # Criterion: Using FocalLoss
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    
    # Model Definition
    model = RecurrentClassifier(
        input_size=input_shape[-1], 
        hidden_size=hidden_size_hp,
        num_layers=num_layers_hp,
        num_classes=num_classes,
        dropout_rate=dropout_hp,
        bidirectional=bidirectional_hp,
        rnn_type=rnn_type_hp
    ).to(device)

    # --- APPLY INITIALIZATION (NEW) ---
    initialize_weights(model, init_scheme=init_scheme_hp) 

    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=lr_hp, 
        weight_decay=weight_decay_hp 
    )
    
    # Scheduler Definition (ReduceLROnPlateau)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',
        factor=scheduler_factor_hp, 
        patience=scheduler_patience_hp, 
        verbose=False,
        threshold=1e-4,
        min_lr=1e-7
    )

    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))
    weight_max_norm_hp = trial.suggest_float("weight_max_norm", low=0.1, high=6.0, log=False)
    # --- 5. Run Training ---
    try:
        _, _, best_val_f1 = fit(
            model=model,
            train_loader=train_loader, 
            val_loader=val_loader,     
            epochs=EPOCHS,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            l1_lambda=l1_lambda_hp, # <-- PASSED THE TUNED L1 PARAMETER
            l2_lambda=0,            # L2 still handled by optimizer
            patience=PATIENCE,
            evaluation_metric="val_f1",
            mode='max',
            restore_best_weights=True,
            writer=None,
            verbose=1,
            experiment_name=f"optuna_trial_{trial.number}",
            trial=trial,
            scheduler=scheduler,
            weight_max_norm = weight_max_norm_hp
        )
        
        return best_val_f1

    except optuna.exceptions.TrialPruned:
        return 0.0 
    except Exception as e:
        print(f"Trial {trial.number} failed with exception: {e}")
        return 0.0 

# --- 6. Create and Run the Optuna Study ---
print("--- Starting Optuna Hyperparameter Tuning ---")

study = optuna.create_study(
    direction="maximize", 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_startup_trials=3)
)

study.optimize(objective, n_trials=50) 

print("\n--- Tuning Complete ---")
print(f"Best trial number: {study.best_trial.number}")
print(f"Best validation F1-score: {study.best_value:.4f}")
print("Best hyperparameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2025-11-11 18:10:42,454] A new study created in memory with name: no-name-c53587af-d312-4612-a90a-a0ced4ea6c97


--- Starting Optuna Hyperparameter Tuning ---
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.0623, F1 Score=0.5395 | Val: Loss=0.9702, F1 Score=0.7004
Epoch   2/200 | Train: Loss=0.9431, F1 Score=0.6549 | Val: Loss=0.8782, F1 Score=0.7680
Epoch   3/200 | Train: Loss=0.8805, F1 Score=0.6725 | Val: Loss=0.9064, F1 Score=0.5692
Epoch   4/200 | Train: Loss=0.7388, F1 Score=0.7660 | Val: Loss=0.7867, F1 Score=0.6723
Epoch   5/200 | Train: Loss=0.6812, F1 Score=0.7988 | Val: Loss=0.7588, F1 Score=0.8145
Epoch   6/200 | Train: Loss=0.6269, F1 Score=0.8270 | Val: Loss=0.7725, F1 Score=0.8586
Epoch   7/200 | Train: Loss=0.5755, F1 Score=0.8460 | Val: Loss=0.6446, F1 Score=0.8667
Epoch   8/200 | Train: Loss=0.5258, F1 Score=0.8554 | Val: Loss=0.6570, F1 Score=0.8619
Epoch   9/200 | Train: Loss=0.5153, F1 Score=0.8473 | Val: Loss=0.5524, F1 Score=0.8064
Epoch  10/200 | Train: Loss=0.4948, F1 Score=0.8495 | Val: Loss=0.6285, F1 Score=0.8700
Epoch  11/200 | Train: Loss=0.4790, F1 Score=0.8590

[I 2025-11-11 18:11:57,913] Trial 0 finished with value: 0.8921109555660751 and parameters: {'WINDOW': 20, 'STRIDE': 15, 'hidden_size': 128, 'num_layers': 1, 'dropout_rate': 0.13573375074615965, 'rnn_type': 'GRU', 'bidirectional': True, 'init_scheme': 'kaiming_normal', 'lr': 0.0006544276067279963, 'weight_decay': 1.1731213437457893e-06, 'l1_lambda': 5.396709736523594e-07, 'focal_gamma': 0.5, 'scheduler_patience': 4, 'scheduler_factor': 0.5, 'weight_max_norm': 5.616934331600047}. Best is trial 0 with value: 0.8921109555660751.


Epoch  66/200 | Train: Loss=0.1787, F1 Score=0.9320 | Val: Loss=0.4457, F1 Score=0.8780
Early stopping triggered after 66 epochs.
Best model restored from epoch 51 with val_f1 0.8921
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.2406, F1 Score=0.6208 | Val: Loss=1.0796, F1 Score=0.6859
Epoch   2/200 | Train: Loss=1.2257, F1 Score=0.6176 | Val: Loss=1.0758, F1 Score=0.7743
Epoch   3/200 | Train: Loss=1.2097, F1 Score=0.6622 | Val: Loss=1.0582, F1 Score=0.7808
Epoch   4/200 | Train: Loss=1.1881, F1 Score=0.6871 | Val: Loss=1.0429, F1 Score=0.6762
Epoch   5/200 | Train: Loss=1.1522, F1 Score=0.6734 | Val: Loss=1.0021, F1 Score=0.6681
Epoch   6/200 | Train: Loss=1.0963, F1 Score=0.6800 | Val: Loss=0.9345, F1 Score=0.6955
Epoch   7/200 | Train: Loss=1.0589, F1 Score=0.6721 | Val: Loss=0.9163, F1 Score=0.6986
Epoch   8/200 | Train: Loss=1.0356, F1 Score=0.6790 | Val: Loss=0.9625, F1 Score=0.6318
Epoch   9/200 | Train: Loss=1.0133, F1 Score=0.6827 | Val: Loss=0.9168, F1 Score=0.6771
Ep

[I 2025-11-11 18:12:34,546] Trial 1 finished with value: 0.7808408483390261 and parameters: {'WINDOW': 40, 'STRIDE': 10, 'hidden_size': 32, 'num_layers': 3, 'dropout_rate': 0.387892677372777, 'rnn_type': 'GRU', 'bidirectional': True, 'init_scheme': 'orthogonal', 'lr': 7.252627958721836e-05, 'weight_decay': 5.9207657352008226e-05, 'l1_lambda': 3.674561284571719e-05, 'focal_gamma': 0.5, 'scheduler_patience': 10, 'scheduler_factor': 0.1, 'weight_max_norm': 2.694457539247971}. Best is trial 0 with value: 0.8921109555660751.


...

Epoch   8/200 | Train: Loss=0.7241, F1 Score=0.7608 | Val: Loss=0.7273, F1 Score=0.7848
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.0674, F1 Score=0.5692 | Val: Loss=0.9179, F1 Score=0.7867
Epoch   2/200 | Train: Loss=0.9358, F1 Score=0.6376 | Val: Loss=0.9104, F1 Score=0.6618
Epoch   3/200 | Train: Loss=0.8469, F1 Score=0.6730 | Val: Loss=0.8595, F1 Score=0.6413
Epoch   4/200 | Train: Loss=0.7563, F1 Score=0.7408 | Val: Loss=0.6566, F1 Score=0.7978
Epoch   5/200 | Train: Loss=0.5901, F1 Score=0.8278 | Val: Loss=0.5395, F1 Score=0.8584
Epoch   6/200 | Train: Loss=0.5061, F1 Score=0.8481 | Val: Loss=0.6355, F1 Score=0.8738
Epoch   7/200 | Train: Loss=0.4939, F1 Score=0.8653 | Val: Loss=0.5865, F1 Score=0.8153
Epoch   8/200 | Train: Loss=0.4391, F1 Score=0.8626 | Val: Loss=0.6859, F1 Score=0.8780
Epoch   9/200 | Train: Loss=0.4234, F1 Score=0.8662 | Val: Loss=0.5755, F1 Score=0.8612
Epoch  10/200 | Train: Loss=0.3686, F1 Score=0.8818 | Val: Loss=0.4572, F1 Score=0.8371
Epoch  11

[I 2025-11-11 18:38:49,359] Trial 48 finished with value: 0.0 and parameters: {'WINDOW': 40, 'STRIDE': 10, 'hidden_size': 128, 'num_layers': 2, 'dropout_rate': 0.4771443593592058, 'rnn_type': 'GRU', 'bidirectional': True, 'init_scheme': 'orthogonal', 'lr': 0.0003711171759853359, 'weight_decay': 4.276193190162569e-05, 'l1_lambda': 7.080089498939243e-07, 'focal_gamma': 2.5, 'scheduler_patience': 5, 'scheduler_factor': 0.2, 'weight_max_norm': 1.5836593533874728}. Best is trial 27 with value: 0.9499284386309793.


Epoch  36/200 | Train: Loss=0.1068, F1 Score=0.9574 | Val: Loss=0.3157, F1 Score=0.8967
Training 200 epochs...
Epoch   1/200 | Train: Loss=1.0437, F1 Score=0.6147 | Val: Loss=0.9398, F1 Score=0.6178
Epoch   2/200 | Train: Loss=0.9132, F1 Score=0.6578 | Val: Loss=0.9767, F1 Score=0.8213
Epoch   3/200 | Train: Loss=0.8868, F1 Score=0.6757 | Val: Loss=0.8571, F1 Score=0.6447
Epoch   4/200 | Train: Loss=0.7485, F1 Score=0.7393 | Val: Loss=0.8656, F1 Score=0.6679
Epoch   5/200 | Train: Loss=0.6486, F1 Score=0.7958 | Val: Loss=0.5646, F1 Score=0.8449
Epoch   6/200 | Train: Loss=0.6200, F1 Score=0.7957 | Val: Loss=0.6176, F1 Score=0.7882
Epoch   7/200 | Train: Loss=0.5761, F1 Score=0.8101 | Val: Loss=0.5415, F1 Score=0.8375
Epoch   8/200 | Train: Loss=0.4959, F1 Score=0.8431 | Val: Loss=0.5392, F1 Score=0.8500
Epoch   9/200 | Train: Loss=0.4412, F1 Score=0.8641 | Val: Loss=0.5432, F1 Score=0.8912
Epoch  10/200 | Train: Loss=0.4450, F1 Score=0.8717 | Val: Loss=0.4095, F1 Score=0.8749
Epoch  11

[I 2025-11-11 18:40:08,372] Trial 49 finished with value: 0.9387854956235793 and parameters: {'WINDOW': 40, 'STRIDE': 10, 'hidden_size': 128, 'num_layers': 3, 'dropout_rate': 0.523213212530997, 'rnn_type': 'GRU', 'bidirectional': False, 'init_scheme': 'orthogonal', 'lr': 0.0006901340837149368, 'weight_decay': 7.380348679878528e-06, 'l1_lambda': 1.0932095374793297e-06, 'focal_gamma': 3.0, 'scheduler_patience': 5, 'scheduler_factor': 0.2, 'weight_max_norm': 2.2016696892921264}. Best is trial 27 with value: 0.9499284386309793.


Epoch  52/200 | Train: Loss=0.0669, F1 Score=0.9722 | Val: Loss=0.3626, F1 Score=0.9344
Early stopping triggered after 52 epochs.
Best model restored from epoch 37 with val_f1 0.9388

--- Tuning Complete ---
Best trial number: 27
Best validation F1-score: 0.9499
Best hyperparameters found:
  WINDOW: 40
  STRIDE: 10
  hidden_size: 128
  num_layers: 3
  dropout_rate: 0.5377302226330053
  rnn_type: GRU
  bidirectional: True
  init_scheme: orthogonal
  lr: 0.0009785537478176932
  weight_decay: 1.7897780974039973e-05
  l1_lambda: 2.571348226648045e-06
  focal_gamma: 3.5
  scheduler_patience: 5
  scheduler_factor: 0.2
  weight_max_norm: 2.166420382127338


## 🧠 **Model Training**

In [58]:
# Initialize best model tracking variables
best_model = None
best_performance = float('-inf')

In [97]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): The neural network model to train
        train_loader (DataLoader): PyTorch DataLoader containing training data batches
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss)
        optimizer (torch.optim): Optimization algorithm (e.g., Adam, SGD)
        scaler (GradScaler): PyTorch's gradient scaler for mixed precision training
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)
        l1_lambda (float): Lambda for L1 regularization
        l2_lambda (float): Lambda for L2 regularization

    Returns:
        tuple: (average_loss, f1 score) - Training loss and f1 score for this epoch
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm


        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = logits.argmax(dim=1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

In [98]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.

    Args:
        model (nn.Module): The neural network model to evaluate (must be in eval mode)
        val_loader (DataLoader): PyTorch DataLoader containing validation data batches
        criterion (nn.Module): Loss function used to calculate validation loss
        device (torch.device): Computing device ('cuda' for GPU, 'cpu' for CPU)

    Returns:
        tuple: (average_loss, accuracy) - Validation loss and accuracy for this epoch

    Note:
        This function automatically sets the model to evaluation mode and disables
        gradient computation for efficiency during validation.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = logits.argmax(dim=1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_accuracy = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_accuracy

In [99]:
def log_metrics_to_tensorboard(writer, epoch, train_loss, train_f1, val_loss, val_f1, model):
    """
    Log training metrics and model parameters to TensorBoard for visualization.

    Args:
        writer (SummaryWriter): TensorBoard SummaryWriter object for logging
        epoch (int): Current epoch number (used as x-axis in TensorBoard plots)
        train_loss (float): Training loss for this epoch
        train_f1 (float): Training f1 score for this epoch
        val_loss (float): Validation loss for this epoch
        val_f1 (float): Validation f1 score for this epoch
        model (nn.Module): The neural network model (for logging weights/gradients)

    Note:
        This function logs scalar metrics (loss/f1 score) and histograms of model
        parameters and gradients, which helps monitor training progress and detect
        issues like vanishing/exploding gradients.
    """
    # Log scalar metrics
    writer.add_scalar('Loss/Training', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('F1/Training', train_f1, epoch)
    writer.add_scalar('F1/Validation', val_f1, epoch)

    # Log model parameters and gradients
    for name, param in model.named_parameters():
        if param.requires_grad:
            # Check if the tensor is not empty before adding a histogram
            if param.numel() > 0:
                writer.add_histogram(f'{name}/weights', param.data, epoch)
            if param.grad is not None:
                # Check if the gradient tensor is not empty before adding a histogram
                if param.grad.numel() > 0:
                    if param.grad is not None and torch.isfinite(param.grad).all():
                        writer.add_histogram(f'{name}/gradients', param.grad.data, epoch)

In [100]:
import torch
import torch.nn as nn
import optuna
# Importa la funzione di utilità se necessario (anche se lo faremo manualmente) 7
# import torch.nn.utils as nn_utils 

def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode='max',
        restore_best_weights=True, writer=None, verbose=10, experiment_name="",
        trial=None, scheduler=None, weight_max_norm=None):
    """
    Train the neural network model on the training data and validate on the validation data.
    """

    # --- NUOVA FUNZIONE: Normalizzazione/Vincolo dei Pesi ---
    def apply_weight_constraint(model, max_norm):
        with torch.no_grad():
            for name, module in model.named_modules():
                # Applica solo ai layer con pesi (es. Linear, Conv1d, Conv2d)
                if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
                    # Vincola il parametro 'weight'
                    if hasattr(module, 'weight') and module.weight is not None:
                        # Calcola la norma L2 del tensore dei pesi
                        norm = module.weight.norm(2)
                        
                        if norm > max_norm:
                            # Se la norma è maggiore del vincolo, riscala il peso.
                            # Ciò garantisce che la norma L2 sia esattamente 'max_norm' o inferiore.
                            module.weight.data.mul_(max_norm / norm)

    # Initialize metrics tracking
    training_history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': []
    }

    # Initialize best_metric before loop
    best_metric = float('-inf') if mode == 'max' else float('inf')
    best_epoch = 0
    
    if patience > 0:
        patience_counter = 0

    if verbose > 0: # Print only if verbose
        print(f"Training {epochs} epochs...")

    # Main training loop: iterate through epochs
    for epoch in range(1, epochs + 1):

        # Forward pass, compute gradients, update weights
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )
        
        # --- CODICE AGGIUNTO QUI: APPLICA IL VINCOLO SUI PESI ---
        if weight_max_norm is not None and weight_max_norm > 0:
            apply_weight_constraint(model, weight_max_norm)
        # --------------------------------------------------------

        # Evaluate model on validation data
        val_loss, val_f1 = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Store metrics
        training_history['train_loss'].append(train_loss)
        training_history['val_loss'].append(val_loss)
        training_history['train_f1'].append(train_f1)
        training_history['val_f1'].append(val_f1)

        # Write to TensorBoard
        if writer is not None:
            log_metrics_to_tensorboard(
                writer, epoch, train_loss, train_f1, val_loss, val_f1, model
            )

        # Print progress
        if verbose > 0:
            if epoch % verbose == 0 or epoch == 1:
                print(f"Epoch {epoch:3d}/{epochs} | "
                      f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                      f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

        # Get current metric for Pruning & Early Stopping
        current_metric = training_history[evaluation_metric][-1]

        # Scheduler Step
        if scheduler is not None:
            scheduler.step(current_metric)
        
        # Optuna Pruning
        if trial is not None:
            try:
                trial.report(current_metric, epoch)
            except Exception as e:
                # Gestione dell'errore (solo se Optuna è effettivamente usato)
                print(f"Error during Optuna report: {e}") 
                pass

            if trial.should_prune():
                # Pruning requested
                raise optuna.exceptions.TrialPruned()
        # End Pruning

        # Early stopping logic (omesso per brevità, resta invariato)
        if patience > 0:
            is_improvement = (current_metric > best_metric) if mode == 'max' else (current_metric < best_metric)

            if is_improvement:
                best_metric = current_metric
                best_epoch = epoch
                # Non uso la variabile `experiment_name` qui, assumo che sia definita in un contesto più ampio
                torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if verbose > 0:
                        print(f"Early stopping triggered after {epoch} epochs.")
                    break

    # Restore best model weights (omesso per brevità, resta invariato)
    if restore_best_weights and patience > 0 and best_epoch > 0:
        model.load_state_dict(torch.load("models/"+experiment_name+'_model.pt'))
        if verbose > 0:
            print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping (omesso per brevità, resta invariato)
    if patience == 0:
        torch.save(model.state_dict(), "models/"+experiment_name+'_model.pt')
        if mode == 'max':
            best_metric = max(training_history[evaluation_metric])
        else:
            best_metric = min(training_history[evaluation_metric])
            
    if patience > 0 and best_epoch == 0:
         if mode == 'max':
             best_metric = max(training_history[evaluation_metric])
         else:
             best_metric = min(training_history[evaluation_metric])


    # Close TensorBoard writer
    if writer is not None:
        writer.close()

    return model, training_history, best_metric

In [ ]:
# # Create model and display architecture with parameter count
# rnn_model = RecurrentClassifier(
#     input_size=input_shape[-1], # Pass the number of features
#     hidden_size=HIDDEN_SIZE,
#     num_layers=HIDDEN_LAYERS,
#     num_classes=num_classes,
#     dropout_rate=DROPOUT_RATE,
#     bidirectional=BIDIRECTIONAL,
#     rnn_type=RNN_TYPE
#     ).to(device)
# recurrent_summary(rnn_model, input_size=input_shape)

# # Set up TensorBoard logging and save model architecture
# experiment_name = f"{RNN_TYPE}_{BIDIRECTIONAL}"
# writer = SummaryWriter("./"+logs_dir+"/"+experiment_name)
# x = torch.randn(1, input_shape[0], input_shape[1]).to(device)
# writer.add_graph(rnn_model, x)

# # Define optimizer with L2 regularization
# optimizer = torch.optim.AdamW(rnn_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_LAMBDA)

# # Enable mixed precision training for GPU acceleration
# scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

In [ ]:
# %%time
# # Train model and track training history
# rnn_model, training_history = fit(
#     model=rnn_model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     epochs=EPOCHS,
#     criterion=criterion,
#     optimizer=optimizer,
#     scaler=scaler,
#     device=device,
#     writer=writer,
#     verbose=1,
#     experiment_name=f"{RNN_TYPE}_{BIDIRECTIONAL}",
#     patience=20
#     )

# # Update best model if current performance is superior
# if training_history['val_f1'][-1] > best_performance:
#     best_model = rnn_model
#     best_performance = training_history['val_f1'][-1]

In [102]:
# --- 4. Load the Best Model from the Study ---
print("\n--- Loading Best Model from Optuna Study ---")

best_params = study.best_params

# Re-create the best model architecture
best_model = RecurrentClassifier(
    input_size=input_shape[-1],
    hidden_size=best_params["hidden_size"],
    num_layers=best_params["num_layers"],
    num_classes=num_classes,
    dropout_rate=best_params["dropout_rate"],
    bidirectional=best_params["bidirectional"],
    rnn_type=best_params["rnn_type"]
).to(device)

# Load the saved state dict from the best trial
try:
    best_model_path = f"models/optuna_trial_{study.best_trial.number}_model.pt"
    best_model.load_state_dict(torch.load(best_model_path))
    print(f"Successfully loaded best model from {best_model_path}")
    
    # Now you can run the "Plot Confusion Matrix" cell (Cell 34)
    # to evaluate this best_model.

except FileNotFoundError:
    print(f"ERROR: Could not find model file {best_model_path}.")
    print("This might happen if the best trial was pruned before saving a model.")
    print("You may need to re-run the study or check the 'models/' directory.")


--- Loading Best Model from Optuna Study ---
Successfully loaded best model from models/optuna_trial_27_model.pt


## Plot History

In [ ]:
# @title Plot Hitory
# Create a figure with two side-by-side subplots (two columns)
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(18, 5))

# Plot of training and validation loss on the first axis
ax1.plot(training_history['train_loss'], label='Training loss', alpha=0.3, color='#ff7f0e', linestyle='--')
ax1.plot(training_history['val_loss'], label='Validation loss', alpha=0.9, color='#ff7f0e')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot of training and validation accuracy on the second axis
ax2.plot(training_history['train_f1'], label='Training f1', alpha=0.3, color='#ff7f0e', linestyle='--')
ax2.plot(training_history['val_f1'], label='Validation f1', alpha=0.9, color='#ff7f0e')
ax2.set_title('F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

# Adjust the layout and display the plot
plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

## Plot Confusion Matrix


In [ ]:
# @title Plot Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Collect predictions and ground truth labels
val_preds, val_targets = [], []
with torch.no_grad():  # Disable gradient computation for inference
    for xb, yb in val_loader:
        xb = xb.to(device)

        # Forward pass: get model predictions
        logits = rnn_model(xb)
        preds = logits.argmax(dim=1).cpu().numpy()

        # Store batch results
        val_preds.append(preds)
        val_targets.append(yb.numpy())

# Combine all batches into single arrays
val_preds = np.concatenate(val_preds)
val_targets = np.concatenate(val_targets)

# --- Overall Validation Metrics (Weighted) ---
print("--- Overall Validation Metrics (Weighted) ---")
val_acc = accuracy_score(val_targets, val_preds)
val_prec = precision_score(val_targets, val_preds, average='weighted')
val_rec = recall_score(val_targets, val_preds, average='weighted')
val_f1 = f1_score(val_targets, val_preds, average='weighted')
print(f"Accuracy over the validation set: {val_acc:.4f}")
print(f"Precision over the validation set: {val_prec:.4f}")
print(f"Recall over the validation set: {val_rec:.4f}")
print(f"F1 score over the validation set: {val_f1:.4f}")

# --- Metrics Per Class ---
# (Labels based on cell [6] label_map = {'no_pain': 0, 'low_pain': 1, 'high_pain': 2})
target_names = ['no_pain', 'low_pain', 'high_pain']
print("\n--- Metrics Per Class ---")
report = classification_report(val_targets, val_preds, target_names=target_names)
print(report)


# --- Confusion Matrix ---
# Generate confusion matrix for detailed error analysis
cm = confusion_matrix(val_targets, val_preds)

# Create numeric labels for heatmap annotation
labels = np.array([f"{num}" for num in cm.flatten()]).reshape(cm.shape)

# Visualise confusion matrix
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=labels, fmt='',
            cmap='Blues',
            xticklabels=target_names,  # Use class names for x-axis
            yticklabels=target_names   # Use class names for y-axis
           )
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

## Hyperparameter tuning

In [106]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset
from datetime import datetime
import os # For checking if the models directory exists

# --- Setup Assumed Parameters (If not already defined) ---
# NOTE: Replace these with the actual values from your notebook if they are missing
# These values are derived from typical notebook context for this type of problem.
# WINDOW = 40
# STRIDE = 10
# BATCH_SIZE = 64
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_classes = 3
# inverse_label_map = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}

# Set the window/step parameters for test data
SEQ_LENGTH = WINDOW # Should be 40
STEP = STRIDE       # Should be 10

# --- 1. Load the Best Model from the Study ---
print("\n--- Loading Best Model from Optuna Study ---")

try:
    best_params = study.best_params

    # Re-create the best model architecture
    best_model = RecurrentClassifier(
        input_size=input_shape[-1],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        num_classes=num_classes,
        dropout_rate=best_params["dropout_rate"],
        bidirectional=best_params["bidirectional"],
        rnn_type=best_params["rnn_type"]
    ).to(device)

    # Load the saved state dict from the best trial
    best_model_path = f"models/optuna_trial_7_model.pt"
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"Successfully loaded best model from {best_model_path}")

except FileNotFoundError:
    print(f"ERROR: Could not find model file {best_model_path}.")
    print("Please check the 'models/' directory and ensure the training completed successfully.")
    # Exit or raise error if model cannot be loaded
    # raise
except NameError as e:
    print(f"Error: A required variable or class is missing: {e}")
    # Exit or raise error if variables are missing
    # raise
except Exception as e:
    print(f"An unexpected error occurred during model loading: {e}")
    # raise

# --- 2. Define Test Sliding Window Function ---
def create_test_sliding_windows(X_df_full, feature_cols, seq_length, step):
    """
    Creates overlapping sequences (sliding windows) from flat test data, 
    grouped by 'sample_index'.
    """
    X_sequences = []
    sample_index_map = [] # To track which user each window belongs to
    
    grouped = X_df_full.groupby('sample_index')
    
    print(f"Creating test sliding windows (Length={seq_length}, Step={step})...")
    
    for sample_id, user_data in grouped:
        user_features = user_data[feature_cols].values.astype(np.float32)
        total_steps = len(user_features)
        
        for i in range(0, total_steps - seq_length + 1, step):
            window = user_features[i : i + seq_length]
            X_sequences.append(window)
            sample_index_map.append(sample_id)
            
    print(f"Created {len(X_sequences)} total test sequences.")
    return np.array(X_sequences, dtype=np.float32), sample_index_map

# --- 3. Prepare Test Data and DataLoader ---
print("--- Preparing Test Data Loader ---")

X_test_flat = df_public_test.copy()
print(f"Using {len(feature_cols)} features for test data (should be 32).")

# Reshape Test Data into Sequences
X_test_seq, test_index_map = create_test_sliding_windows(
    X_test_flat, 
    feature_cols, 
    SEQ_LENGTH, 
    STEP
)
print(f"Reshaped X_test_seq shape: {X_test_seq.shape}")

# Convert to Tensor & Create Dataset (Test dataset has no labels)
X_test_tensor = torch.from_numpy(X_test_seq)
test_ds = TensorDataset(X_test_tensor)
print(f"\nCreated test TensorDataset with {len(test_ds)} windows.")

# Create Test DataLoader
test_loader = make_loader(
    test_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False,  # No need to shuffle for prediction
    drop_last=False # Must process all test samples
)
print(f"Created test_loader. Batches: {len(test_loader)}")

# --- 4. Generate Predictions ---
print("\n--- Generating Predictions for all windows ---")
best_model.eval()   # Set model to evaluation mode
all_logits = []

with torch.no_grad():
    for (inputs,) in test_loader: # Unpack tuple (test_ds only has inputs)
        inputs = inputs.to(device)
        
        # Use mixed precision for inference if available
        # This line is good practice but assumes `torch.amp.autocast` is available
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = best_model(inputs)
            
        all_logits.append(logits.cpu().numpy())

final_logits_all_windows = np.concatenate(all_logits)
print(f"Generated logits for {len(final_logits_all_windows)} windows.")


# --- 5. Create Submission File (Aggregating Predictions) ---
print("\n--- Aggregating window predictions by averaging logits ---")

# Create a DataFrame to manage aggregation
pred_df = pd.DataFrame({
    'sample_index': test_index_map
})

# Add logit columns
for c in range(num_classes):
    pred_df[f'logit_{c}'] = final_logits_all_windows[:, c]

# Group by user and average the logits
logit_cols = [f'logit_{c}' for c in range(num_classes)]
submission_logits_avg = pred_df.groupby('sample_index')[logit_cols].mean()

# Get the final class by taking argmax of the *averaged* logits
final_numeric_predictions = submission_logits_avg.idxmax(axis=1).str.replace('logit_', '').astype(int)
final_numeric_predictions = final_numeric_predictions.reset_index(name='prediction')

# Map numeric predictions (0, 1, 2) back to string labels
final_labels = final_numeric_predictions['prediction'].map(inverse_label_map)

# Create final submission DataFrame
submission_df = pd.DataFrame({
    'sample_index': final_numeric_predictions['sample_index'],
    'label': final_labels
})

# Save to CSV
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f'submission_{timestamp}.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"\n✅ Successfully saved final submission file: **{submission_filename}** with {len(submission_df)} rows.")
print("\nFirst 5 rows of the submission file:")
print(submission_df.head())


--- Loading Best Model from Optuna Study ---
An unexpected error occurred during model loading: Error(s) in loading state_dict for RecurrentClassifier:
	size mismatch for rnn.weight_ih_l0: copying a param with shape torch.Size([192, 32]) from checkpoint, the shape in current model is torch.Size([384, 32]).
	size mismatch for rnn.weight_hh_l0: copying a param with shape torch.Size([192, 64]) from checkpoint, the shape in current model is torch.Size([384, 128]).
	size mismatch for rnn.bias_ih_l0: copying a param with shape torch.Size([192]) from checkpoint, the shape in current model is torch.Size([384]).
	size mismatch for rnn.bias_hh_l0: copying a param with shape torch.Size([192]) from checkpoint, the shape in current model is torch.Size([384]).
	size mismatch for rnn.weight_ih_l0_reverse: copying a param with shape torch.Size([192, 32]) from checkpoint, the shape in current model is torch.Size([384, 32]).
	size mismatch for rnn.weight_hh_l0_reverse: copying a param with shape torch.

## Predict Public tests

In [104]:
# --- 1. Check for necessary variables ---
# This code assumes 'best_model', 'df_public_test', 'feature_cols', 
# 'WINDOW', 'STRIDE', 'BATCH_SIZE', 'device', 'num_classes', and 'make_loader'
# already exist in your notebook's memory from the previous cells.

# *** NEW: Window parameters must match training (from cell [13]) ***
SEQ_LENGTH = WINDOW # Should be 40
STEP = STRIDE     # Should be 10

if 'best_model' not in locals():
    print("Error: 'best_model' not found.")
    print("Please make sure you have run the training cell and 'best_model' is in memory.")
else:
    print("--- Preparing Test Data Loader ---")

    # --- 2. Use the loaded and processed test data ---
    # We use df_public_test, which was loaded in [5] and processed in [7] & [8]
    X_test_flat = df_public_test.copy()

    # --- 3. Handle NaNs in test data ---
    # (This is already done by the preProcess and scaling cells [7] and [8])

    # --- 4. Select the same feature columns ---
    # (feature_cols was defined in cell [12])
    print(f"Using {len(feature_cols)} features for test data (should be 32).")

    # --- 5. Reshape Test Data into Sequences ---
    
    # *** NEW: Sliding Window Function for Test Set ***
    # This function matches the logic used in 'build_sequences' (cell [14])
    # for generating sliding windows (13 windows per 160 steps).
    def create_test_sliding_windows(X_df_full, feature_cols, seq_length, step):
        """
        Creates overlapping sequences (sliding windows) from flat test data, 
        grouped by 'sample_index'.
        
        Returns:
            - np.array: The 3D sequence data (num_windows, seq_length, num_features)
            - list: A list of 'sample_index' for each window, to map predictions back.
        """
        X_sequences = []
        sample_index_map = [] # To track which user each window belongs to
        
        # Group by user
        grouped = X_df_full.groupby('sample_index')
        
        print(f"Creating test sliding windows (Length={seq_length}, Step={step})...")
        
        for sample_id, user_data in grouped:
            # Ensure data is float32, matching training data (cell [10])
            user_features = user_data[feature_cols].values.astype(np.float32)
            
            # Total time steps for this user (should be 160)
            total_steps = len(user_features)
            
            # Iterate and create windows
            # (Starts: 0, 10, ... 120. 13 windows total)
            for i in range(0, total_steps - seq_length + 1, step):
                window = user_features[i : i + seq_length]
                X_sequences.append(window)
                sample_index_map.append(sample_id) # Store the user ID for this window
                
        print(f"Created {len(X_sequences)} total test sequences.")
        # Ensure final array is float32
        return np.array(X_sequences, dtype=np.float32), sample_index_map

    # We pass the full X_test_flat dataframe (which has 'sample_index')
    X_test_seq, test_index_map = create_test_sliding_windows(
        X_test_flat, 
        feature_cols, 
        SEQ_LENGTH, 
        STEP
    )
    print(f"Reshaped X_test_seq shape: {X_test_seq.shape}")

    # --- 6. Convert to Tensor & Create Dataset ---
    # Use torch.from_numpy, same as in cell [17] for training
    X_test_tensor = torch.from_numpy(X_test_seq)
    test_ds = TensorDataset(X_test_tensor) # Test dataset has no labels
    print(f"\nCreated test TensorDataset with {len(test_ds)} windows.")

    # --- 7. Create Test DataLoader ---
    # make_loader (cell [11] or [19]) is available.
    test_loader = make_loader(
        test_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=False,  # No need to shuffle for prediction
        drop_last=False   # Must process all test samples
    )
    print(f"Created test_loader. Batches: {len(test_loader)}")

    # --- 8. Generate Predictions ---
    print("\n--- Generating Predictions for all windows ---")
    best_model.eval()  # Set model to evaluation mode
    all_logits = []

    with torch.no_grad():
        for (inputs,) in test_loader: # Unpack tuple (test_ds only has inputs)
            inputs = inputs.to(device)
            
            # Use mixed precision for inference if available
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = best_model(inputs)
                
            all_logits.append(logits.cpu().numpy())

    final_logits_all_windows = np.concatenate(all_logits)
    print(f"Generated logits for {len(final_logits_all_windows)} windows.")

    # --- 9. Create Submission File (Aggregating Predictions) ---
    print("\n--- Aggregating window predictions by averaging logits ---")
    
    # Create a DataFrame to manage aggregation
    pred_df = pd.DataFrame({
        'sample_index': test_index_map
    })
    
    # Add logit columns
    for c in range(num_classes): # num_classes should be 3 (from cell [16])
        pred_df[f'logit_{c}'] = final_logits_all_windows[:, c]

    # Group by user and average the logits
    logit_cols = [f'logit_{c}' for c in range(num_classes)]
    submission_logits_avg = pred_df.groupby('sample_index')[logit_cols].mean()
    
    # Get the final class by taking argmax of the *averaged* logits
    final_numeric_predictions = submission_logits_avg.idxmax(axis=1).str.replace('logit_', '').astype(int)
    final_numeric_predictions = final_numeric_predictions.reset_index(name='prediction')
    
    # Map numeric predictions (0, 1, 2) back to string labels
    # (This map is defined in cell [6] of the notebook)
    inverse_label_map = {0: 'no_pain', 1: 'low_pain', 2: 'high_pain'}
    final_labels = final_numeric_predictions['prediction'].map(inverse_label_map)
    
    # Create submission DataFrame
    submission_df = pd.DataFrame({
        'sample_index': final_numeric_predictions['sample_index'],
        'label': final_labels
    })

    from datetime import datetime
    # Save to CSV (Kaggle environment standard name)
    submission_df.to_csv(f'try{datetime.now()}submission.csv', index=False)
    print(f"\nSuccessfully saved 'submission.csv' with {len(submission_df)} rows.")
    print(submission_df.head())

--- Preparing Test Data Loader ---
Using 32 features for test data (should be 32).
Creating test sliding windows (Length=40, Step=10)...
Created 17212 total test sequences.
Reshaped X_test_seq shape: (17212, 40, 32)

Created test TensorDataset with 17212 windows.
Created test_loader. Batches: 269

--- Generating Predictions for all windows ---
Generated logits for 17212 windows.

--- Aggregating window predictions by averaging logits ---

Successfully saved 'submission.csv' with 1324 rows.
   sample_index    label
0             0  no_pain
1             1  no_pain
2             2  no_pain
3             3  no_pain
4             4  no_pain
